# 03 — Preprocessing

Select features, engineer venue-distance features, log-transform the target, and split train/test. **Encoding (OneHot) and scaling (StandardScaler) live inside the sklearn `Pipeline` built in nb04**, not here — that way the saved `rent_pipeline.pkl` can `.predict()` on raw input without the caller having to reproduce a cleaning chain by hand.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

df = pd.read_csv('../data/processed/listings_eda.csv')
print(df.shape)
print(df.columns.tolist())

(2566, 34)
['price', 'accommodates', 'bedrooms', 'bathrooms', 'neighbourhood_cleansed', 'room_type', 'property_type', 'minimum_nights', 'latitude', 'longitude', 'availability_365', 'number_of_reviews', 'reviews_per_month', 'host_is_superhost', 'host_response_rate', 'host_acceptance_rate', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'n_amenities', 'has_ac', 'has_pool', 'has_hot_tub', 'has_gym', 'has_dishwasher', 'has_washer', 'has_dryer', 'has_free_parking', 'has_elevator', 'has_balcony']


In [2]:
# Keep only useful features
AMENITY_FLAGS = ['has_ac', 'has_pool', 'has_hot_tub', 'has_gym',
                 'has_dishwasher', 'has_washer', 'has_dryer',
                 'has_free_parking', 'has_elevator', 'has_balcony']

REVIEW_SCORES = ['review_scores_rating', 'review_scores_accuracy',
                 'review_scores_cleanliness', 'review_scores_checkin',
                 'review_scores_communication', 'review_scores_location',
                 'review_scores_value']

HOST_FEATURES = ['host_is_superhost', 'host_response_rate', 'host_acceptance_rate']

FEATURES = ['accommodates', 'bedrooms', 'bathrooms',
            'neighbourhood_cleansed', 'room_type', 'property_type',
            'minimum_nights', 'latitude', 'longitude',
            'availability_365', 'number_of_reviews', 'reviews_per_month',
            'n_amenities'] + AMENITY_FLAGS + HOST_FEATURES + REVIEW_SCORES
TARGET = 'price'

df = df[FEATURES + [TARGET]].copy()
df = df.dropna(subset=[TARGET])
df = df[df[TARGET] > 0]
print(f'After price filter: {df.shape}')

After price filter: (2566, 34)


## Event venue proximity features
Distance (km) from each listing to major Zurich event venues.
These vary per listing via lat/lng and act as a proxy for event-driven demand.

In [3]:
# Haversine distance (km) — no external library needed
def haversine_km(lat, lon, vlat, vlon):
    R = 6371.0
    lat, lon, vlat, vlon = map(np.radians, [lat, lon, vlat, vlon])
    a = np.sin((vlat-lat)/2)**2 + np.cos(lat)*np.cos(vlat)*np.sin((vlon-lon)/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Major Zurich event venues (lat, lon)
VENUES = {
    "dist_hallenstadion_km": (47.4317, 8.5495),   # indoor arena
    "dist_letzigrund_km":    (47.3794, 8.5033),   # football stadium
    "dist_messe_km":         (47.4100, 8.5486),   # exhibition centre
    "dist_opernhaus_km":     (47.3654, 8.5464),   # opera house
    "dist_hb_km":            (47.3779, 8.5403),   # main station / event hub
}

for col, (vlat, vlon) in VENUES.items():
    df[col] = haversine_km(df["latitude"].values, df["longitude"].values, vlat, vlon)

df["min_dist_venue_km"] = df[[c for c in VENUES]].min(axis=1)
print("Venue distance features added:")
print(df[[*VENUES, "min_dist_venue_km"]].describe().round(2))

Venue distance features added:
       dist_hallenstadion_km  dist_letzigrund_km  dist_messe_km  \
count                2566.00             2566.00        2566.00   
mean                    6.27                3.11           4.20   
std                     2.01                1.58           1.73   
min                     0.47                0.11           0.11   
25%                     5.32                1.89           3.16   
50%                     6.60                2.98           4.48   
75%                     7.64                4.40           5.38   
max                    12.24                8.16           9.88   

       dist_opernhaus_km  dist_hb_km  min_dist_venue_km  
count            2566.00     2566.00            2566.00  
mean                2.77        2.39               1.30  
std                 1.66        1.33               0.82  
min                 0.18        0.08               0.08  
25%                 1.49        1.36               0.72  
50%              

In [4]:
# NA handling and encoding are now done inside the sklearn Pipeline in nb04
# (SimpleImputer + OneHotEncoder). We just write the raw, engineered features here.
print(f'Rows after feature selection + price filter: {len(df)}')
print(f'NA counts (handled by SimpleImputer in nb04):')
print(df.isna().sum()[df.isna().sum() > 0])

Rows after feature selection + price filter: 2566
NA counts (handled by SimpleImputer in nb04):
bedrooms                         7
bathrooms                        9
reviews_per_month              614
host_response_rate             213
host_acceptance_rate           174
review_scores_rating           614
review_scores_accuracy         614
review_scores_cleanliness      614
review_scores_checkin          614
review_scores_communication    614
review_scores_location         614
review_scores_value            614
dtype: int64


In [5]:
X = df.drop(columns=[TARGET])
y = np.log1p(df[TARGET])  # log-transform: stabilises right-skewed price distribution

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'\nColumn dtypes in X_train:')
print(X_train.dtypes.value_counts())

Train: (2052, 39), Test: (514, 39)

Column dtypes in X_train:
float64    20
int64      16
object      3
Name: count, dtype: int64


In [6]:
os.makedirs('../data/processed', exist_ok=True)
# Saved with categoricals as plain strings — the Pipeline in nb04 handles encoding.
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)
print('Splits saved (raw features, no encoding / no scaling).')

Splits saved (raw features, no encoding / no scaling).
